# Deploying Iris-detection model using Vertex AI


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginica**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

## Get started

### Install Vertex AI SDK for Python and other required packages



In [1]:

# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform

### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [2]:
PROJECT_ID = "nifty-harmony-474217-q6"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [3]:
BUCKET_URI = f"gs://mlops-course-nifty-harmony-474217-q6-unique"  # @param {type:"string"}

**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [4]:
! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

Creating gs://mlops-course-nifty-harmony-474217-q6-unique/...
ServiceException: 409 A Cloud Storage bucket named 'mlops-course-nifty-harmony-474217-q6-unique' already exists. Try another name. Bucket names must be globally unique across all Google Cloud projects, including those outside of your organization.


### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [5]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [6]:
import os
import sys

### Configure resource names

Set a name for the following parameters:

`MODEL_ARTIFACT_DIR` - Folder directory path to your model artifacts within a Cloud Storage bucket, for example: "my-models/fraud-detection/trial-4"

`REPOSITORY` - Name of the Artifact Repository to create or use.

`IMAGE` - Name of the container image that is pushed to the repository.

`MODEL_DISPLAY_NAME` - Display name of Vertex AI model resource.

In [7]:
MODEL_ARTIFACT_DIR = "my-models/iris-classifier-week-1"  # @param {type:"string"}
REPOSITORY = "iris-classifier-repo"  # @param {type:"string"}
IMAGE = "iris-classifier-img"  # @param {type:"string"}
MODEL_DISPLAY_NAME = "iris-classifier"  # @param {type:"string"}

# Set the defaults if no names were specified
if MODEL_ARTIFACT_DIR == "[your-artifact-directory]":
    MODEL_ARTIFACT_DIR = "custom-container-prediction-model"

if REPOSITORY == "[your-repository-name]":
    REPOSITORY = "custom-container-prediction"

if IMAGE == "[your-image-name]":
    IMAGE = "sklearn-fastapi-server"

if MODEL_DISPLAY_NAME == "[your-model-display-name]":
    MODEL_DISPLAY_NAME = "sklearn-custom-container"

# Homework Pipeline

## Requirement 1: Importing data from the Google Storage Bucket

In [8]:
from google.cloud import storage
import pandas as pd

TRAINING_DATA_BUCKET_NAME="training_data_mlops_w1"
TRAINING_BLOB="iris.csv"

def load_iris_from_gc(bucket_name=TRAINING_DATA_BUCKET_NAME,blob_name=TRAINING_BLOB):
    client=storage.Client()
    bucket=client.bucket(bucket_name)
    blob=bucket.blob(blob_name)
    data_bytes=blob.download_as_bytes()
    df=pd.read_csv(pd.io.common.BytesIO(data_bytes))
    return df

iris_df=load_iris_from_gc()
data=iris_df
iris_df.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


## Simple Decision Tree model
Build a Decision Tree model on iris data

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

Exception ignored in: <finalize object at 0x7f03787830a0; dead>
Traceback (most recent call last):
  File "/opt/conda/lib/python3.10/weakref.py", line 586, in __call__
    def __call__(self, _=None):
KeyboardInterrupt: 


In [10]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [11]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


In [12]:
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

### Storing output artifact in the artifact bucket

In [13]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
!gsutil cp artifacts/model.joblib gs://mlops-course-nifty-harmony-474217-q6-unique/my-models/iris-classifier-week-1/{timestamp}/model.joblib
print(f"gs://training_data_mlops_w1/artifacts/{timestamp}/model.joblib")


^C
gs://training_data_mlops_w1/artifacts/20251012_112521/model.joblib


### Inference Script

#### imports

In [14]:
from google.cloud import storage
import joblib
import os

In [15]:
{BUCKET_URI}

{'gs://mlops-course-nifty-harmony-474217-q6-unique'}

In [16]:
from google.cloud import storage
import joblib

bucket_name = "mlops-course-nifty-harmony-474217-q6-unique"
model_dir = "my-models/iris-classifier-week-1/"

client = storage.Client()
bucket = client.bucket(bucket_name)
blobs = list(bucket.list_blobs(prefix=model_dir))
model_files = [blob.name for blob in blobs if blob.name.endswith("model.joblib")]
model_files.sort()
latest_model_path = model_files[-1]

blob = bucket.blob(latest_model_path)
blob.download_to_filename("latest_model.joblib")

model = joblib.load("latest_model.joblib")
print("Latest model loaded")


Latest model loaded


In [17]:
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


# Week 2

In [23]:
#!pip install dvc

In [19]:
#!git init
#!dvc init
#!dvc remote add -d myremote gs://training_data_mlops_w1

Reinitialized existing Git repository in /home/jupyter/.git/
ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.
^C
ERROR: interrupted by the user


In [20]:
#!dvc remote add -d -f myremote gs://training_data_mlops_w1

Setting 'myremote' as a default remote.
^C


In [24]:
!git add Homework_Pipeline.ipynb

In [25]:
!dvc add iris.csv

 ⠋ Checking graph
Adding...                                                                       
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Checking out /home/jupyter/iris.csv   0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|1/1 [00:00, 26.65file/s]

To track the changes with git, run:

	git add iris.csv.dvc

To enable auto staging, run:

	dvc config core.autostage true


In [26]:
#!git add .gitignore iris.csv.dvc

In [ ]:
#!git commit -m "Clean tracking: Git for code, DVC for data"

## Augmenting IRIS Data

In [27]:
# Get existing data
import pandas as pd
from sklearn.datasets import load_iris

iris_df_v1=load_iris_from_gc()
iris_df_v1.head()

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa


In [28]:
len(iris_df_v1)

150

In [29]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

In [30]:
data=iris_df_v1
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [31]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


### Saving files

In [32]:
iris_df_v1.to_csv("iris.csv",index=False)

In [33]:
import pickle
import joblib

joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

### Adding files to DVC and GIT

In [ ]:
!dvc add artifacts/model.joblib iris.csv
!git add artifacts/.gitignore artifacts/model.joblib.dvc iris.csv.dvc Homework_Pipeline.ipynb
!git commit -m "Trained model 1"
!git tag -a "v1.0"

 ⠋ Checking graph
  0% Adding...|             | artifacts/model.joblib |0/2 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Checking out /home/jupyter/artifacts/m0/1 [00:00<?,    ?files/s]
  0% Adding...|                           | iris.csv |0/2 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Checking out /home/jupyter/iris.csv   0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|2/2 [00:00, 34.68file/s]

To track the changes with git, run:

	git add artifacts/model.joblib.dvc iris.

## Add rows

In [ ]:
df=load_iris(as_frame=True) #from load_iris
new_rows=df.frame.sample(10,replace=True)
new_rows['target'] = new_rows['target'].map({i: name for i, name in enumerate(df.target_names)})
new_rows=new_rows.rename(columns={'sepal length (cm)':'sepal_length',
                                  'sepal width (cm)':'sepal_width',
                                 'petal length (cm)':'petal_length',
                                 'petal width (cm)':'petal_width',
                                 'target':'species'})
iris_df_v2=pd.concat([iris_df_v1,new_rows])
iris_df_v2.tail()

In [ ]:
len(iris_df_v2)

In [ ]:
iris_df_v2.to_csv("iris.csv",index=False)
!dvc add iris.csv

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pandas.plotting import parallel_coordinates
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn import metrics

In [ ]:
data=iris_df_v2
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

In [ ]:
mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

In [ ]:
iris_df_v2.to_csv("iris.csv",index=False)

In [ ]:
!dvc add artifacts/model.joblib
!dvc add iris.csv
!git add artifacts/.gitignore artifacts/model.joblib.dvc Homework_Pipeline.ipynb iris.csv.dvc
!git commit -m "Trained model 2"
!git tag -a "v2.0"

In [ ]:
git checkout v1.0
dvc checkout